# 05-6. 날짜와 시간

## Goal

시간대가 있는 ISO 8601 값을 UTC로 통일하고 오류 행을 나눈다.


## Setup

`fixtures/05-text-processing/timestamp-events.jsonl`를 읽는다. 저장소 루트에서 JupyterLab을 실행한다.


In [ ]:
from pathlib import Path
import sys


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("requirements.txt가 있는 저장소 루트에서 JupyterLab을 실행하세요.")


ROOT = find_project_root()
FIXTURE_DIR = ROOT / "fixtures" / "05-text-processing"

assert sys.version_info >= (3, 10)
assert FIXTURE_DIR.is_dir()

print("Python:", sys.version.split()[0])
print("실습 데이터:", FIXTURE_DIR)


import json

fixture_path = FIXTURE_DIR / "timestamp-events.jsonl"
records = [json.loads(line) for line in fixture_path.read_text(encoding="utf-8").splitlines()]
records


## Steps

`parse_utc(value)`를 완성한다. `Z`를 처리하고 naive datetime은 거부한다.


In [ ]:
from datetime import datetime, timezone

TODO_DONE = False


def parse_utc(value: str) -> datetime:
    raise NotImplementedError


## Checks

TODO를 구현한 뒤 `TODO_DONE = True`로 바꾸고 공개 경계 검증을 실행한다. 검증이 통과해도 다른 입력이 모두 올바르다는 보장은 아니다.


In [ ]:
if not TODO_DONE:
    print("TODO를 구현한 뒤 TODO_DONE을 True로 바꾸세요.")
else:
    assert parse_utc("2026-08-14T10:30:00+09:00").isoformat() == "2026-08-14T01:30:00+00:00"
    assert parse_utc("2026-08-14T01:30:00Z").tzinfo == timezone.utc
    try:
        parse_utc("2026-08-14T01:30:00")
    except ValueError:
        pass
    else:
        raise AssertionError("naive datetime을 허용했습니다.")
    for invalid in ("", "   ", 123):
        try:
            parse_utc(invalid)
        except (TypeError, ValueError):
            pass
        else:
            raise AssertionError("타입·빈 값 경계를 허용했습니다.")

    fixture_valid, fixture_errors = [], []
    for record in records:
        try:
            fixture_valid.append({
                **record,
                "timestamp_utc": parse_utc(record["timestamp"]),
            })
        except (KeyError, TypeError, ValueError) as error:
            fixture_errors.append({
                "event": record.get("event"),
                "error": error,
            })
    assert len(fixture_valid) == 2 and len(fixture_errors) == 2
    assert all(item["timestamp_utc"].tzinfo == timezone.utc for item in fixture_valid)
    assert [
        item["event"]
        for item in sorted(fixture_valid, key=lambda item: item["timestamp_utc"])
    ] == ["login", "api"]
    assert {item["event"] for item in fixture_errors} == {"naive", "invalid"}
    print("공개 경계 검증 통과: fixture 정상 2건 / 오류 2건")


## Next Steps

저장·비교용 UTC 값과 표시용 로컬 시간을 나눈다.
